# Скрапинг сотрудников ВШЭ

## 📥 Импорт библиотек

In [ ]:
import re
import requests
from lxml import html
from urllib.parse import urljoin, urlparse
import pandas as pd
from tqdm.notebook import tqdm
from pprint import pprint
import datetime
import json
import time
import os

Раскомментировать, если чего-то не хватает

In [ ]:
# !pip install requests lxml pandas tqdm

## ⚙️ Вспомогательные функции для парсинга

### Создание DOM дерева

In [ ]:
def clean_text(s):
    if not s:
        return None
    return re.sub(r"\s+", " ", s).strip()

def make_tree(html_text):
    return html.fromstring(html_text)

### Вспомогательные ф-ии для выделения блоков `sidebar` и `main` из дерева

In [ ]:
def get_sidebar_root(tree):
    if tree is None:
        return None
    side_nodes = tree.xpath("//div[@class='l-extra js-mobile_menu_content is-desktop']/div")
    return side_nodes[0] if side_nodes else None

def get_main_root(tree):
    if tree is None:
        return None
    main_nodes = tree.xpath("//div[contains(@class, 'main__inner')]")
    return main_nodes[0] if main_nodes else None

### Ф-ия для получения `person-id`, который потом используется в API публикациях

In [ ]:
def get_person_id(tree, url: str | None = None) -> int | None:
    if tree is not None:
        vals = tree.xpath("//script[@data-person-id]/@data-person-id")

        for v in vals:
            s = str(v).strip()
            if s.isdigit():
                return int(s)
            try:
                pid = int(s)
                return pid
            except ValueError:
                continue

    if url:
        try:
            path = urlparse(url).path.rstrip("/")
            if path:
                last = path.split("/")[-1]
                if last.isdigit():
                    return int(last)
        except Exception:
            pass

    return None

### Блок `Sidebar` - колонка справа

#### `avatar`: url фото

In [ ]:
def parse_avatar(tree, base_url: str = "https://www.hse.ru"):
    avatar = None
    side_el = get_sidebar_root(tree)
    if side_el is not None:
        avatar_div = side_el.xpath(".//div[contains(@class,'person-avatar')]")
        if avatar_div:
            style = avatar_div[0].get("style", "") or ""
            m = re.search(r"url\(([^)]+)\)", style)
            if m:
                raw_avatar = m.group(1).strip().strip("'\"")
                avatar = urljoin(base_url, raw_avatar)
    return avatar

#### `languages`: список языков

In [ ]:
def parse_languages(tree):
    languages = []
    side_el = get_sidebar_root(tree)
    if side_el is not None:
        langs = side_el.xpath(
            ".//dl[contains(@class,'main-list-language-knowledge-level')]//dd/text()"
        )
        languages = [clean_text(x) for x in langs if clean_text(x)]
    return languages

#### `contacts`: телефоны, адреса, кабинеты, часы приема, расписание

In [ ]:
def parse_contacts(tree, base_url: str = "https://www.hse.ru"):
    contacts = {
        "phones": None,
        "address": None,
        "hours": None,
        "timetable_url": None,
    }

    side_el = get_sidebar_root(tree)
    if side_el is None:
        return contacts

    # блок контакты
    contacts_dl = side_el.xpath(
        ".//dl[contains(@class,'main-list')][dt[contains(text(),'Контакты')]]"
    )
    if contacts_dl:
        dl = contacts_dl[0]
        dds = dl.xpath("./dd")

        # телефоны
        if dds:
            phone_dd = dds[0]
            texts = [clean_text(t) for t in phone_dd.xpath(".//text()") if clean_text(t)]
            phones = []
            for t in texts:
                if t.startswith("Телефон"):
                    continue
                phones.append(t)
            if phones:
                contacts["phones"] = " | ".join(phones)

        # адрес + часы
        addr_dd = dl.xpath(".//dd[contains(@class,'address-with-hours')]")
        if addr_dd:
            addr_text = clean_text(" ".join(addr_dd[0].xpath(".//text()")))
            if "Время присутствия:" in addr_text:
                left, right = addr_text.split("Время присутствия:", 1)
                address = clean_text(left.replace("Адрес:", ""))
                hours = clean_text(right)
            elif "Время консультаций:" in addr_text:
                left, right = addr_text.split("Время консультаций:", 1)
                address = clean_text(left.replace("Адрес:", ""))
                hours = clean_text(right)
            else:
                address = addr_text.replace("Адрес:", "").strip()
                hours = None
            contacts["address"] = address or None
            contacts["hours"] = hours or None

    # расписание
    timetable_dl = side_el.xpath(
        ".//dl[contains(@class,'person-extra-indent-timetable')]"
    )
    if timetable_dl:
        link = timetable_dl[0].xpath(".//a[@class='link']/@href")
        if link:
            contacts["timetable_url"] = urljoin(base_url, link[0])

    return contacts

#### `research_ids`: ссылки на страницы профиля с ресёрчом

In [ ]:
def parse_research_ids(tree, base_url: str = "https://www.hse.ru"):
    research_ids = {}

    side_el = get_sidebar_root(tree)
    if side_el is None:
        return research_ids

    ids_dl = side_el.xpath(
        ".//dl[contains(@class,'person-extra-indent') "
        "     and not(contains(@class,'person-extra-indent-timetable')) "
        "     and not(contains(@class,'colleagues'))]"
    )
    if ids_dl:
        for dd in ids_dl[0].xpath("./dd"):
            label = clean_text("".join(dd.xpath(".//span[@class='b']/text()")))

            if not label:
                link = dd.xpath(".//a")
                if link:
                    name = clean_text(link[0].text_content())
                    url = link[0].get("href")
                    url = urljoin(base_url, url) if url else None
                    research_ids[name] = {"label": name, "url": url}
                continue

            link = dd.xpath(".//a")
            if link:
                value_text = clean_text(link[0].text_content())
                url = link[0].get("href")
                url = urljoin(base_url, url) if url else None
            else:
                full = clean_text(dd.text_content())
                value_text = full.replace(label, "").replace(":", "").strip()
                url = None

            research_ids[label] = {
                "value": value_text,
                "url": url,
            }

    return research_ids

#### `managers`: руководители / заместители

In [ ]:
def parse_managers(tree, base_url: str = "https://www.hse.ru"):
    managers = []
    side_el = get_sidebar_root(tree)
    if side_el is not None:
        coll_dl = side_el.xpath(".//dl[contains(@class,'colleagues')]")
        if coll_dl:
            for dd in coll_dl[0].xpath("./dd"):
                name = clean_text("".join(dd.xpath(".//a/text()")))
                url = dd.xpath(".//a/@href")
                url = urljoin(base_url, url[0]) if url else None
                role = clean_text("".join(dd.xpath(".//span[contains(@class,'grey')]/text()")))
                managers.append({
                    "name": name,
                    "url": url,
                    "role": role,
                })

    return managers

### Блок 'About' — всё основное про сотрудника

#### `full_name` (ФИО)

In [ ]:
def parse_full_name(tree):
    main_el = get_main_root(tree)
    full_name = None
    if main_el is not None:
        name_el = main_el.xpath(".//h1[contains(@class,'person-caption')]/text()")
        full_name = clean_text(name_el[0]) if name_el else None
    return full_name

#### `employment`: список назначений и связанных подразделений

In [ ]:
def parse_employment(tree, base_url: str = "https://www.hse.ru"):
    main_el = get_main_root(tree)
    employment = []

    if main_el is not None:
        ul_list = main_el.xpath(".//ul[contains(@class,'employment-add')]")
        if ul_list:
            ul = ul_list[0]
            for li in ul.xpath("./li"):
                title = clean_text("".join(
                    li.xpath(".//span[contains(@class,'person-appointment-title')]/text()")
                ))

                units = []
                for a in li.xpath(".//a[@class='link']"):
                    unit_name = clean_text(a.text_content())
                    href = a.get("href")
                    if href:
                        href = urljoin(base_url, href)
                    units.append({
                        "name": unit_name,
                        "url": href,
                    })

                employment.append({
                    "title": title,
                    "units": units,
                })

    return employment

#### `employment_traits`: короткие статусы/звания

In [ ]:
def parse_employment_traits(tree):
    main_el = get_main_root(tree)
    traits = []

    if main_el is not None:
        traits_ul = main_el.xpath(".//ul[contains(@class,'employment-traits')]")
        if traits_ul:
            for li in traits_ul[0].xpath("./li"):
                txt = clean_text(li.text_content())
                if txt:
                    traits.append(txt)

    return traits

#### `employment_addition`: доп. фразы про трудоустройство

In [ ]:
def parse_employment_addition(tree):
    main_el = get_main_root(tree)
    addition = []

    if main_el is not None:
        add_ul = main_el.xpath(".//ul[contains(@class,'person-employment-addition')]")
        if add_ul:
            for li in add_ul[0].xpath("./li"):
                txt = clean_text(li.text_content())
                if txt:
                    addition.append(txt)

    return addition

#### `degrees`: образование и учёные степени

In [ ]:
def parse_degrees(tree):
    main_el = get_main_root(tree)
    degrees = []

    if main_el is not None:
        degree_blocks = main_el.xpath(
            ".//div[contains(@class,'b-person-data') and @tab-node='sci-degrees1']"
        )

        for block in degree_blocks:
            entries = block.xpath(
                ".//div[contains(@class,'g-list_closer')]//div[contains(@class,'with-indent')]"
            )
            for entry in entries:
                year_el = entry.xpath(".//div[contains(@class,'person-list-hangover')]/text()")
                year_raw = clean_text(year_el[0]) if year_el else None

                year = None
                if year_raw:
                    try:
                        year = int(year_raw)
                    except Exception:
                        year = None

                full_text = clean_text(entry.text_content())
                if year_raw and full_text.startswith(year_raw):
                    text = clean_text(full_text[len(year_raw):])
                else:
                    text = full_text

                if text:
                    degrees.append({
                        "year": year,
                        "text": text,
                    })

    return degrees

#### `professional_interests`: профессиональные интересы (теги)

In [ ]:
def parse_professional_interests(tree, base_url: str = "https://www.hse.ru"):
    main_el = get_main_root(tree)
    interests = []

    if main_el is not None:
        interest_blocks = main_el.xpath(
            ".//div[contains(@class,'b-person-data') and @tab-node='sci-intrests']"
        )

        for block in interest_blocks:
            for a in block.xpath(".//a[contains(@class,'tag')]"):
                label = clean_text(a.text_content())
                href = a.get("href")
                if href:
                    href = urljoin(base_url, href)
                if label:
                    interests.append({
                        "label": label,
                        "url": href,
                    })

    return interests

#### `extra_education`: доп. образование / курсы / стажировки

In [ ]:
def parse_extra_education(tree):
    main_el = get_main_root(tree)
    extra_ed = []

    if main_el is not None:
        extra_blocks = main_el.xpath(
            ".//div[contains(@class,'b-person-data') and @tab-node='additional_education']"
        )

        for block in extra_blocks:
            year_paragraphs = block.xpath(".//p[strong]")
            for p in year_paragraphs:
                year_raw = clean_text("".join(p.xpath(".//strong/text()")))
                year = None
                if year_raw:
                    try:
                        year = int(year_raw)
                    except Exception:
                        year = None

                ul = p.xpath("./following-sibling::ul[1]")
                if not ul:
                    continue
                ul = ul[0]

                for li in ul.xpath(".//li"):
                    text = clean_text(li.text_content())
                    if not text:
                        continue
                    extra_ed.append({
                        "year": year,
                        "text": text,
                    })

    return extra_ed

#### `awards`: достижения и поощрения

In [ ]:
def parse_awards(tree):
    main_el = get_main_root(tree)
    awards = []

    if main_el is not None:
        award_blocks = main_el.xpath(
            ".//div[contains(@class,'b-person-data') and @tab-node='awards']"
        )

        for block in award_blocks:
            for li in block.xpath(".//ul[contains(@class,'g-list')]/li"):
                txt = clean_text(li.text_content())
                if not txt:
                    continue
                awards.append(txt)

    return awards

#### `theses`: Выпускные квалификационные работы (ВКР)

In [ ]:
def parse_theses(tree, base_url: str = "https://www.hse.ru"):
    theses = []

    # Позже доделаю - они тоже через API подгружаются

    return theses

#### `courses`: Учебные курсы

In [ ]:
def parse_courses(tree, base_url: str = "https://www.hse.ru"):
    main_el = get_main_root(tree)
    courses = []

    if main_el is None:
        return courses

    course_blocks = main_el.xpath(
        ".//div[contains(@class,'edu-courses')]"
    )

    for block in course_blocks:
        academic_year = None
        h2 = block.xpath(".//h2/text()")
        if h2:
            h2_text = clean_text(h2[0])
            m_year = re.search(r"(\d{4}/\d{4})", h2_text)
            if m_year:
                academic_year = m_year.group(1)

        for li in block.xpath(".//ul[contains(@class,'g-list')]/li"):
            archive_toggle = li.xpath(".//span[contains(@class,'edu-courses-archive-toogle')]")
            if archive_toggle:
                continue

            a = li.xpath(".//a[@class='link' and contains(@href, '/edu/courses/')]")
            if not a:
                continue

            link_el = a[0]
            title = clean_text(link_el.text_content())
            href = link_el.get("href")
            url = urljoin(base_url, href) if href else None

            lang_el = li.xpath(".//span[contains(@class,'language-label')]/text()")
            language = clean_text(lang_el[0]) if lang_el else None

            li_text = clean_text(li.text_content())

            meta = None
            m_meta = re.search(r"\(([^()]*)\)", li_text)
            if m_meta:
                meta = clean_text(m_meta.group(1))

            courses.append(
                {
                    "title": title,
                    "url": url,
                    "academic_year": academic_year,
                    "language": language,
                    "meta": meta,
                }
            )

    return courses

#### `grants`: гранты

In [ ]:
def parse_grants(tree):
    main_el = get_main_root(tree)
    grants = []

    if main_el is None:
        return grants

    grant_blocks = main_el.xpath(
        ".//div[contains(@class,'b-person-data') and @tab-node='grants']"
    )

    for block in grant_blocks:
        for li in block.xpath(".//ol/li | .//ul/li"):
            txt = clean_text(li.text_content())
            if not txt:
                continue

            grant_number = None
            m_num = re.search(r"Номер:\s*([^,]+)", txt)
            if m_num:
                grant_number = clean_text(m_num.group(1))

            years = None
            year_matches = re.findall(r"(\d{4})\s*г", txt)
            if year_matches:
                try:
                    if len(year_matches) == 1:
                        y = int(year_matches[0])
                        years = {"start": y, "end": y}
                    else:
                        years = {
                            "start": int(year_matches[0]),
                            "end": int(year_matches[-1]),
                        }
                except Exception:
                    years = None

            grants.append(
                {
                    "text": txt,
                    "number": grant_number,
                    "years": years,
                }
            )

    return grants

#### `editorial_staff`: Участие в редколлегиях научных журналов

In [ ]:
def parse_editorial_staff(tree):
    main_el = get_main_root(tree)
    editorial_staff = []

    if main_el is None:
        return editorial_staff

    blocks = main_el.xpath(
        ".//div[contains(@class,'b-person-data') and @tab-node='editorial-staff']"
    )
    if not blocks:
        return {"editorial_staff": editorial_staff}

    block = blocks[0]

    for div in block.xpath(".//div[contains(@class,'with-indent')]"):
        txt = clean_text(div.text_content())
        if not txt:
            continue

        start_year = None
        m_year = re.search(r"(\d{4})\s*г", txt)
        if m_year:
            try:
                start_year = int(m_year.group(1))
            except Exception:
                start_year = None

        journal = None
        m_journal = re.search(r"«([^»]+)»", txt)
        if m_journal:
            journal = clean_text(m_journal.group(1))

        editorial_staff.append(
            {
                "text": txt,
                "start_year": start_year,
                "journal": journal,
            }
        )

    return editorial_staff

#### `conferences`: Конференции

In [ ]:
def parse_conferences(tree, base_url: str = "https://www.hse.ru"):
    main_el = get_main_root(tree)
    conferences = []

    if main_el is None:
        return conferences

    blocks = main_el.xpath(
        ".//div[contains(@class,'b-person-data') and @tab-node='conferences']"
    )
    if not blocks:
        return conferences

    block = blocks[0]

    last_year = None

    for ul in block.xpath(".//ul[contains(@class,'g-list_closer')]"):
        for li in ul.xpath(".//li[contains(@class,'li2')]"):
            year_el = li.xpath(".//div[contains(@class,'person-list-hangover')]/text()")
            if year_el:
                year_raw = clean_text(year_el[0])
                year = None
                if year_raw:
                    try:
                        year = int(year_raw)
                    except Exception:
                        year = None
                last_year = year
            else:
                year = last_year

            p = li.xpath(".//p")[0] if li.xpath(".//p") else li
            description = clean_text(p.text_content())

            links = []
            for a in li.xpath(".//a[@href]"):
                href = a.get("href")
                if href:
                    href = urljoin(base_url, href)
                link_text = clean_text(a.text_content()) or None
                links.append(
                    {
                        "url": href,
                        "text": link_text,
                    }
                )

            if description:
                conferences.append(
                    {
                        "year": year,
                        "description": description,
                        "links": links,
                    }
                )

    return conferences

#### `publiclications`: Публикации, статьи, книги

Фетчинг через API публикаций по id персоны (по 15 штук чтобы не блокернули)

In [ ]:
def fetch_publications_page(person_id: int, page: int = 1, count: int = 15):
    url = "https://publications.hse.ru/api/searchPubs"

    filter_params = (
        f"\"acceptLanguage\":\"ru\"|"
        f"\"pubsAuthor\": {person_id}|"
        f"\"widgetName\": \"AuthorSearch\""
    )

    payload = {
        "type": "ANY",
        "filterParams": filter_params,
        "paginationParams": {
            "publsSort": ["YEAR_DESC", "TITLE_ASC"],
            "publsCount": count,
            "pageId": page,
        },
    }

    headers = {
        "Accept": "application/json, text/plain, */*",
        "Content-Type": "application/json;charset=utf-8",
        "Referer": "https://www.hse.ru/",
    }

    resp = requests.post(url, json=payload, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data["result"]

Итерация по доступным публикациям

In [ ]:
def fetch_all_publications_for_person(
    person_id: int,
    per_page: int = 15,
    max_pages: int | None = None,
):

    all_items = []
    page = 1
    total = None

    while True:
        result = fetch_publications_page(person_id, page=page, count=per_page)

        items = result.get("items", [])
        all_items.extend(items)

        if total is None:
            total = result.get("total")

        more = result.get("more", False)
        remaining = result.get("remaining", 0)

        if not more or remaining <= 0 or not items:
            break

        if max_pages is not None and page >= max_pages:
            break

        page += 1

    return all_items, total

#### `work_experience`: опыт работы

In [ ]:
def parse_work_experience(tree):
    main_el = get_main_root(tree)
    work_experience = []

    if main_el is None:
        return work_experience

    exp_blocks = main_el.xpath(
        ".//div[contains(@class,'b-person-data') and @tab-node='experience']"
    )

    for block in exp_blocks:
        for div in block.xpath(".//div[contains(@class,'with-indent')]"):
            txt = clean_text(div.text_content())
            if not txt:
                continue
            work_experience.append(txt)

    return work_experience

#### `news`: новости

вспомогательная ф-ия для даты

In [ ]:
RU_MONTHS = {
    "янв.": 1,
    "фев.": 2,
    "мар.": 3,
    "апр.": 4,
    "май": 5,
    "июнь": 6,
    "июль": 7,
    "авг.": 8,
    "сент.": 9,
    "окт.": 10,
    "нояб.": 11,
    "дек.": 12,
}


def parse_news_date(day_str: str, month_str: str, year_str: str):
    day_str = clean_text(day_str)
    month_str = clean_text(month_str).lower()
    year_str = clean_text(year_str)

    if not day_str or not month_str or not year_str:
        return None

    try:
        day = int(day_str)
        year = int(year_str)
    except ValueError:
        return None

    month = RU_MONTHS.get(month_str)
    if month is None:
        return None

    try:
        d = datetime.date(year, month, day)
        return d.isoformat()  # "YYYY-MM-DD"
    except ValueError:
        return None

ф-ия для парсинга новостей

In [ ]:
def parse_news(tree, base_url: str = "https://www.hse.ru"):
    main_el = get_main_root(tree)
    news = []

    if main_el is None:
        return news

    news_blocks = main_el.xpath(
        ".//div[contains(@class,'b-person-data') and @tab-node='press_links_news']"
    )

    for block in news_blocks:
        for post in block.xpath(".//div[contains(@class,'post')]"):
            day_el = post.xpath(".//div[contains(@class,'post-meta__day')]/text()")
            month_el = post.xpath(".//div[contains(@class,'post-meta__month')]/text()")
            year_el = post.xpath(".//div[contains(@class,'post-meta__year')]/text()")

            day = clean_text(day_el[0]) if day_el else None
            month = clean_text(month_el[0]) if month_el else None
            year = clean_text(year_el[0]) if year_el else None

            iso_date = None
            if day and month and year:
                iso_date = parse_news_date(day, month, year)

            link_el = post.xpath(".//div[contains(@class,'post__content')]//h2//a")[0] \
                if post.xpath(".//div[contains(@class,'post__content')]//h2//a") else None

            title = clean_text(link_el.text_content()) if link_el is not None else None
            url = None
            if link_el is not None:
                href = link_el.get("href")
                if href:
                    url = urljoin(base_url, href)

            snippet_p = post.xpath(".//div[contains(@class,'post__text')]//p")[0] \
                if post.xpath(".//div[contains(@class,'post__text')]//p") else None
            snippet = clean_text(snippet_p.text_content()) if snippet_p is not None else None

            if not (title or snippet or url):
                continue

            news.append(
                {
                    "title": title,
                    "url": url,
                    "date": iso_date,
                    "day": day,
                    "month": month,
                    "year": year,
                    "snippet": snippet,
                }
            )

    return news

### Ф-ия сбора полного объекта по человеку из HTML

In [ ]:
def parse_person(tree, person_url, base_url: str = "https://www.hse.ru"):

    if tree is None:
        return None

    # ID
    person_id = get_person_id(tree, url=person_url)

    # Identity
    full_name = parse_full_name(tree)
    avatar = parse_avatar(tree, base_url=base_url)
    languages = parse_languages(tree) or []

    # Контакты / менеджеры / research-ids
    contacts = parse_contacts(tree, base_url=base_url) or {}
    managers = parse_managers(tree, base_url=base_url) or []
    research_ids = parse_research_ids(tree, base_url=base_url) or {}

    # Позиция, должности, и тд
    employment = parse_employment(tree, base_url=base_url) or []
    employment_traits = parse_employment_traits(tree) or []
    employment_addition = parse_employment_addition(tree) or []

    # Образование / награды / опыт
    degrees = parse_degrees(tree) or []
    professional_interests = parse_professional_interests(tree, base_url=base_url) or []
    extra_education = parse_extra_education(tree) or []
    awards = parse_awards(tree) or []
    work_experience = parse_work_experience(tree) or []

    # Teaching
    theses = parse_theses(tree, base_url=base_url) or []
    courses = parse_courses(tree, base_url=base_url) or []

    # Research (кроме публикаций)
    grants = parse_grants(tree) or []
    editorial_staff = parse_editorial_staff(tree) or []
    conferences = parse_conferences(tree, base_url=base_url) or []

    # Новости
    news = parse_news(tree, base_url=base_url) or []

    # Публикации по API
    publications = []
    publications_total = 0
    if person_id is not None:
        try:
            pubs, total = fetch_all_publications_for_person(person_id, per_page=50)
            publications = pubs
            publications_total = total if total is not None else len(pubs)
        except Exception:
            publications = []
            publications_total = 0

    
    # Собираем Dict
    person = {
        "meta": {
            "person_id": person_id,
            "parsed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "research_ids": research_ids,
        },

        "identity": {
            "full_name": full_name,
            "avatar": avatar,
            "languages": languages,
        },

        "contacts": contacts,

        "positions": {
            "employment": employment,
            "employment_traits": employment_traits,
            "employment_addition": employment_addition,
            "managers": managers,
        },

        "education": {
            "degrees": degrees,
            "extra_education": extra_education,
            "awards": awards,
            "professional_interests": professional_interests,
        },

        "experience": {
            "work_experience": work_experience,
        },

        "teaching": {
            "theses": theses,
            "courses": courses,
        },

        "research": {
            "grants": grants,
            "editorial_staff": editorial_staff,
            "conferences": conferences,
            "publications_total": publications_total,
            "publications": publications,
        },

        "news": news,
    }

    return person

Обертка для `parse_person`

In [ ]:
def parse_person_by_url(url: str, base_url: str = "https://www.hse.ru", session: requests.Session | None = None):
    sess = session or requests.Session()
    resp = sess.get(url, timeout=20)
    resp.raise_for_status()

    html_text = resp.text
    tree = make_tree(html_text)

    person = parse_person(tree, person_url=url, base_url=base_url)
    if person is not None:
        person.setdefault("meta", {})
        person["meta"]["profile_url"] = url

    return person

### Функция для запуска краулера по персонам

In [ ]:
def short(s, n: int) -> str:
    if not s:
        return ""
    s = str(s)
    return s if len(s) <= n else s[: n - 1] + "…"


def make_checkpoint_path(output_path: str | None) -> str | None:
    if not output_path:
        return None
    if output_path.endswith(".json"):
        return output_path[:-5] + "-checkpoint.json"
    return output_path + "-checkpoint.json"


def make_payload(persons: list[dict]) -> dict:
    return {
        "meta": {
            "scraped_at": datetime.datetime.now(
                datetime.timezone.utc
            ).isoformat(),
            "count": len(persons),
        },
        "persons": persons,
    }


def save_payload(persons: list[dict], path: str) -> None:
    payload = make_payload(persons)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def update_bar_with_person(bar, person: dict) -> None:
    identity = person.get("identity", {}) or {}
    full_name = identity.get("full_name") or "<?>"
    if isinstance(full_name, dict):
        full_name = (
            full_name.get("ru")
            or full_name.get("en")
            or str(full_name)
        )

    positions_block = person.get("positions", {}) or {}
    employment = positions_block.get("employment") or []
    position = ""
    if employment:
        first_emp = employment[0] or {}
        position = (
            first_emp.get("position")
            or first_emp.get("title")
            or first_emp.get("role")
            or ""
        )

    research = person.get("research", {}) or {}
    pubs_total = research.get("publications_total")
    pubs_display = pubs_total if pubs_total is not None else "-"

    bar.set_postfix(
        {
            "имя": short(full_name, 30),
            "должность": short(position, 26),
            "публикации": pubs_display,
        }
    )


def scrape_hse_persons(
    urls,
    limit: int | None = None,
    base_url: str = "https://www.hse.ru",
    sleep: float = 0.0,
    output_path: str | None = None,
    checkpoint_every: int | None = None,
):
    if limit is not None:
        urls = list(urls)[:limit]

    session = requests.Session()
    persons: list[dict] = []

    bar = tqdm(urls, desc="Собираем HSE профили", unit="профиль")

    checkpoint_path = make_checkpoint_path(output_path)

    for idx, url in enumerate(bar, start=1):
        try:
            person = parse_person_by_url(url, base_url=base_url, session=session)
        except Exception as e:
            bar.write(f"[WARN] failed to parse {url}: {e!r}")
            if sleep > 0:
                time.sleep(sleep)
            continue

        if person is not None:
            persons.append(person)
            update_bar_with_person(bar, person)

        if (
            output_path is not None
            and checkpoint_path is not None
            and checkpoint_every is not None
            and checkpoint_every > 0
            and idx % checkpoint_every == 0
        ):
            try:
                save_payload(persons, checkpoint_path)
                bar.write(
                    f"[checkpoint] сохранено {len(persons)} персон → {checkpoint_path}"
                )
            except Exception as e:
                bar.write(f"[WARN] не удалось сохранить чекпойнт: {e!r}")

        if sleep > 0:
            time.sleep(sleep)

    if output_path is not None:
        save_payload(persons, output_path)

        if checkpoint_path is not None and os.path.exists(checkpoint_path):
            try:
                os.remove(checkpoint_path)
            except Exception as e:
                print(f"[WARN] не удалось удалить чекпойнт {checkpoint_path}: {e!r}")

    return persons

## ⚠️ Начало запуска краулера по кампусам и алфавиту

###  Базовые ссылки и HTTP-сессия

In [ ]:
BASE_URL = "https://www.hse.ru"
START_URL = "https://www.hse.ru/org/persons/"

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/129.0 Safari/537.36"
    )
})

print("Загружаем стартовую страницу...")
resp = session.get(START_URL)
resp.raise_for_status()
tree = html.fromstring(resp.text)
print("✅ Краулер готов к работе")

### Сбор кампусов (id + название города)

In [ ]:
campus_lis = tree.xpath("//div[contains(@class, 'filter_topunits')]//li[@hse-value]")

campuses = []
for li in campus_lis:
    campus_id = li.get("hse-value")
    raw_text = " ".join(li.xpath(".//text()"))
    raw_text = " ".join(raw_text.split())
    name = raw_text
    if ":" in name:
        name = name.split(":", 1)[1].strip()
    campuses.append({
        "campus_id": campus_id,
        "campus_name": name,
    })

df_campuses = pd.DataFrame(campuses)
print(f"✅ Найдено кампусов: {len(df_campuses)}")
df_campuses

### Сбор шаблонов ссылок алфавита фамилий

In [ ]:
print("Собираем шаблоны ссылок на алфавит фамилий на стартовой странице...")
letter_nodes = tree.xpath("//div[contains(@class, 'abc-filter__letter')]//a")

letter_templates = []
for a in letter_nodes:
    href = a.get("href")
    url = urljoin(START_URL, href)
    letter_text = "".join(a.xpath(".//text()"))
    letter_text = letter_text.strip()
    letter_templates.append({
        "template_url": url,
        "letter": letter_text,
    })

print(f"✅ Найдено шаблонов букв: {len(letter_templates)}")
letter_templates[:3]

### Обход кампусов и букв, сбор профилей

In [ ]:
def replace_udept(url: str, new_udept: str) -> str:
    if "udept=" not in url:
        return url
    return re.sub(r"udept=\d+", f"udept={new_udept}", url)

In [ ]:
rows_links = []
seen_pairs = set()

print("Обходим кампусы и алфавит, собираем ссылки профилей...")
for campus in tqdm(campuses, desc="Кампусы", unit="кампус"):
    campus_id = campus["campus_id"]
    campus_name = campus["campus_name"]
    campus_new_count = 0
    campus_letters = 0

    for lt in tqdm(letter_templates, desc=f"{campus_name}: буквы", unit="буква", leave=False):
        letter_tmpl = lt["template_url"]
        letter = lt["letter"]
        letter_url = replace_udept(letter_tmpl, campus_id)
        campus_letters += 1

        try:
            r = session.get(letter_url)
            r.raise_for_status()
        except Exception as e:
            tqdm.write(f"⚠️ {campus_name}: ошибка {letter_url}: {e}")
            continue

        t = html.fromstring(r.text)
        person_hrefs = t.xpath("//div[contains(@class, 'content__person-text')]//a/@href")
        full_urls = [urljoin(BASE_URL, href) for href in person_hrefs]

        for profile_url in full_urls:
            key = (campus_id, profile_url)
            if key in seen_pairs:
                continue
            seen_pairs.add(key)
            rows_links.append({
                "campus_id": campus_id,
                "campus_name": campus_name,
                "letter": letter,
                "profile_url": profile_url,
            })
            campus_new_count += 1

    tqdm.write(
        f"📍 {campus_name}: обработано {campus_letters} страниц, новых профилей: {campus_new_count}"
    )

df_links = pd.DataFrame(rows_links)
print(f"\n✅ Всего профилей: {len(df_links)}")
df_links.sample(n=10)

## 🚀 Запуск краулера по преподавателям

- `LIMIT` - количество собираемых профилей (`None`, если собираем все)
- `SLEEP` - таймауты между запросами (в секундах)
- `CHECKPOINT` - каждый Checkpoint -  сохранения файла
- `OUTPUT_PATH`- путь к сохраняемому `.json`

In [ ]:
LIMIT = 30 # или None
SLEEP = 0.1
CHECKPOINT = 10
OUTPUT_PATH = "hse_persons.json"

urls = df_links["profile_url"].tolist()

In [ ]:
persons = scrape_hse_persons(
    urls=urls,
    limit=LIMIT,                       
    sleep=SLEEP,
    checkpoint_every=10,
    output_path=OUTPUT_PATH,
)

print(len(persons))
print("✅ Краулер закончил работу")